This notebook is designed to load neutron data and any non-neutron data (i.e. pressure, current, etc.) in an experimental folder, time bin it, and export it to CSV.

## Initialization

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil

import matplotlib as mpl
import mpl_toolkits.mplot3d as plot3d
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
from scipy import interpolate
from scipy.fft import rfft, rfftfreq
from scipy.optimize import curve_fit

from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.processing.figure_of_merit import gaussian
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df[bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df[BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: NeutronStrategyFactory,
    window_type: WindowType,
    loading: bool,
    settings: NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data


## Experiment ID Input

In [ ]:
# isotope_exps = [247, 251, 257, 258]
isotope_exps = [20, 70, 95, 35, 97]
isotope_exps = [f"ID-383.{id}" for id in isotope_exps]
# isotopes = ["Cs-137", "Co-60", "AmBe-241", "Eu-152"]
isotopes = ["Na-22", "Cs-137", "Eu-152", "Co-60", "K-40"]
bg_exp = "ID-383.1"
experiment_ids = [*isotope_exps, bg_exp]
isotope_name_lookup = {id: name for name, id in zip(isotopes, isotope_exps)}

In [ ]:
calib_input = helpers.get_input_with_default(
    "Do you want to use new calibration? [y/n, or press Enter for yes]",
    "y",
    str
)

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = (
    ExperimentDataKey.NEW_CALIBRATION
    if is_new_calibration
    else ExperimentDataKey.CAEN_CALIBRATION
)

In [ ]:
detector_code = helpers.get_input_required(
    """\
Which detector was used?
1: Original detector (detector 1)
2: New detector (detector 2)
""",
    [Detector.ZERO, Detector.ONE],
    lambda x: Detector(int(x)-1)
)

In [ ]:
default_fit_input = 2  # changed to peak finder mode, approved by Fatima 2024-07-18
fit_input = helpers.get_input_with_default(
    """\
Which bimodal fit type do you want to use?
1: Bounds based
2: Peak finder based (default)
Press Enter for default
""",
    default_fit_input,
    int
)

fit_styles: dict[int, SliceFitStyle] = {
    1: "bounds",
    2: "peak_finder"
}
fit_style = fit_styles.get(fit_input, fit_styles[default_fit_input])

In [ ]:
# kind of window (Nasa, N distribution)
# load or generate
# specific settings for each condition to make namedtuple
# - generator settings (i.e. sigma, etc.)
# - file path prefix for loading
done = False
strategy_factory = NeutronStrategyFactory()

while not done:
    window_input = helpers.get_input_with_default(
        """\
Which neutron classification window do you want to use?
1: NASA window (default)
2: Neutron distribution window
Press Enter for default
""",
        1,
        int
    )
    load_window_input = helpers.get_input_with_default(
        """\
Do you want to load the borders from the standard border file?
[y/n, or press Enter for no]
""",
        "n",
        str
    )
    done = True
    will_load = load_window_input == "y"

    try:
        if window_input == 1:
            if will_load:
                settings = get_nasa_loading_settings(calib_key=calib_key)
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "nasa", True, settings
                )
            else:
                settings = get_nasa_generation_settings(calib_key=calib_key)
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "nasa", False, settings
                )
                pass
        elif window_input == 2:
            if will_load:
                settings = get_n_distro_loading_settings(calib_key=calib_key)
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "n_distro", True, settings
                )
            else:
                settings = get_n_distro_generation_settings()
                factory_fn = make_strategy_factory_fn(
                    strategy_factory, "n_distro", False, settings
                )
        else:
            print("Invalid classification window type given, please try again")
            done = False
    except ValueError as err:
        print("Problem found:")
        print(err)
        print("Please try again")
        done = False

experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}
experiment_neutron_data = make_strategy_for_experiments(experiment_neutron_data, factory_fn)

In [ ]:
bin_length = helpers.get_input_with_default(
    "Enter bin length (in seconds), or press Enter for default (300 s)",
    300,
    int
)
bin_string = f"{bin_length}S"

In [ ]:
analysis_timestamp = datetime.now().strftime("%Y-%m%b-%d-%H-%M-%S")
overall_settings = {
    'calibration_type': repr(calib_key),
    'fitting_style': fit_style,
    'window_settings': repr(settings),
    'bin_length': bin_length
}

In [ ]:
analysis_timestamp

## Data Loading and Initial Processing

### Neutron Data Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    print(exp_id)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load_parquet_psd(exp_id)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = recalibrate(unclassified_df, detector_code)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Get experiment start time
for exp_name, data_dict in experiment_neutron_data.items():
    exp_root = get_exp_root(exp_name)
    with open(exp_root / 'exp_info.toml') as exp_info:
        exp_start_line = [line for line in exp_info if "exp_start" in line][0]
    exp_start_text = exp_start_line.replace("exp_start = ", "").strip()
    exp_start = datetime.fromisoformat(exp_start_text).astimezone(timezone.utc)
    data_dict[ExperimentDataKey.START_TIME] = exp_start

In [ ]:
# Get timetag as clock time
for exp_name, data_dict in experiment_neutron_data.items():
    unclassified = data_dict[ExperimentDataKey.UNCLASSIFIED]
    exp_start = data_dict[ExperimentDataKey.START_TIME]

    unclassified = calculate_event_time(unclassified, exp_start)

    data_dict[ExperimentDataKey.PSD_REPORT] = unclassified

In [ ]:
list(experiment_neutron_data.keys())

In [ ]:
# TODO subtract background
# BG starts 11:48, ends 12:48 (actual beam on 12:55, but using easier time)
data_dict = experiment_neutron_data[bg_exp]
psd_report = data_dict[ExperimentDataKey.UNCLASSIFIED]
# start_time = data_dict[ExperimentDataKey.START_TIME] + timedelta(minutes=30)
# end_time = start_time + timedelta(hours=1)
start_time = data_dict[ExperimentDataKey.START_TIME]
end_time = start_time + timedelta(minutes=20)
print(start_time)
background = psd_report.query("EVENT_TIME.between(@start_time, @end_time)")
experiment_neutron_data["background"] = {}
experiment_neutron_data["background"]["data"] = background

In [ ]:
energy_width = 2.5e-3
background_dict = experiment_neutron_data["background"]
psd_report = background_dict["data"]
Z, xe, ye = get_psd_energy_histogram(
    psd_report,
    calibrated_energy_column,
    energy_width=energy_width,
    psd_bin_count=256
)
background_dict[ExperimentDataKey.PSD_HISTOGRAM] = Z
background_dict[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
background_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye

In [ ]:
# Generate histogram
background_dict = experiment_neutron_data["background"]
bg_x_edges = background_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]

for exp_id in isotope_exps:
    exp_data = experiment_neutron_data[exp_id]
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_bins=bg_x_edges,
        psd_bin_count=256
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye

In [ ]:
# BG histogram
background_dict = experiment_neutron_data["background"]
background = background_dict["data"]
bins = np.linspace(0, 4000, num=401)
Z, edges = np.histogram(background["ENERGY"], bins=bins)
background_dict["histogram"] = Z
background_dict["edges"] = edges

In [ ]:
background_dict = experiment_neutron_data["background"]
bg_histogram = background_dict["histogram"]
bg_edges = background_dict["edges"]
bg_psd_histogram = background_dict[ExperimentDataKey.PSD_HISTOGRAM]
for exp_id in isotope_exps:
    data_dict = experiment_neutron_data[exp_id]
    psd_report = data_dict[ExperimentDataKey.UNCLASSIFIED]
    # data_start_time = datetime.fromisoformat("2024-06-25 13:36:00.000000-07:00")
    # data_end_time = datetime.fromisoformat("2024-06-25 14:36:00.000000-07:00")
    # start_time = data_dict[ExperimentDataKey.START_TIME] + timedelta(minutes=30)
    # end_time = start_time + timedelta(hours=1)
    start_time = data_dict[ExperimentDataKey.START_TIME]
    end_time = start_time + timedelta(minutes=20)
    print(end_time)
    # beam_on = psd_report.query("EVENT_TIME.between(@data_start_time, @data_end_time)")
    beam_on = psd_report.query("EVENT_TIME.between(@start_time, @end_time)")
    Z, _ = np.histogram(psd_report["ENERGY"], bins=bg_edges)
    data_dict["histogram"] = Z
    data_dict["histogram_no_bg"] = Z - bg_histogram
    data_dict["psd_histogram_no_bg"] = data_dict[ExperimentDataKey.PSD_HISTOGRAM] - bg_psd_histogram

In [ ]:
bg_dict = experiment_neutron_data["background"]
edges = bg_dict["edges"]
midpoints = (edges[1:] + edges[:-1]) / 2
xs = np.linspace(edges[0], edges[-1], len(midpoints)+len(edges))
bg_dict["midpoints"] = midpoints
bg_dict["deriv_x"] = xs

In [ ]:
# lowpass filter
from scipy.signal import butter, lfilter, freqz

def butter_lowpass(cutoff, fs, order=5):
    return butter(order, cutoff, fs=fs, btype="low", analog=False)

def butter_lowpass_filter(data, cutoff, fs, order=5):
    b, a = butter_lowpass(cutoff, fs, order=order)
    y = lfilter(b, a, data)
    return y

In [ ]:
midpoints = experiment_neutron_data["background"]["midpoints"]
deriv_x = experiment_neutron_data["background"]["deriv_x"]

order = 6
cutoff = 0.02
period = midpoints[1] - midpoints[0]
fs = 1/period

deriv_cutoff = 0.005
deriv_period = deriv_x[1] - deriv_x[0]
deriv_fs = 1/deriv_period

for exp_id in isotope_exps:
    data_dict = experiment_neutron_data[exp_id]
    histogram = data_dict["histogram_no_bg"]
    histo_filtered = butter_lowpass_filter(histogram, cutoff, fs, order)
    
    histo_interp = interpolate.CubicSpline(midpoints, histo_filtered)
    histo_deriv = histo_interp.derivative()(deriv_x)
    histo_deriv_filtered = butter_lowpass_filter(histo_deriv, deriv_cutoff, deriv_fs, order)

    data_dict["histogram_filtered"] = histo_filtered
    data_dict["histogram_interp"] = histo_interp
    data_dict["histogram_deriv"] = histo_deriv
    data_dict["histogram_deriv_filtered"] = histo_deriv_filtered

In [ ]:
# bg_dict = experiment_neutron_data["background"]
# xs = bg_dict["deriv_x"]
# order = 6
# period = xs[1] - xs[0]
# fs = 1/period  # sample rate, Hz
# cutoff = 0.005  # desired cutoff frequency of the filter, Hz
# bg_dict["filter_params"] = (cutoff, fs, order)

# for exp_id in isotope_exps:
#     data_dict = experiment_neutron_data[exp_id]
#     Z_interp = data_dict["histogram_interp"]
#     # Z = Z_interp(xs)
#     Z_deriv = Z_interp.derivative()(xs)
#     Z_lowpass = butter_lowpass_filter(Z_deriv, cutoff, fs, order)
#     data_dict["histogram_deriv"] = Z_deriv
#     data_dict["histogram_deriv_lowpass"] = Z_lowpass
#     # ax_top.plot(xs, Z)

### Plotting

In [ ]:
vaporwave_colors = [
    [255, 255, 255],
    [128, 69, 229],
    [75,127,255],
    [0,255,255],
    [255,186,129],
    [255,209,86],
    [252,120,183]
]
vaporwave_colors = [[value/255 for value in color] for color in vaporwave_colors]
vaporwave = mpl.colors.LinearSegmentedColormap.from_list("vaporwave", vaporwave_colors, N=256)
# vaporwave = vaporwave.resampled(256)
vaporwave

In [ ]:
# Zs = [bg_removed_histo, bg_histo, psd_histo]
# Zs = [Z.ravel() for Z in Zs]
# Zmin = min([Z.min() for Z in Zs])
# Zmax = max([Z.max() for Z in Zs])
# norm = mpl.colors.Normalize(vmin=Zmin, vmax=Zmax)

# sc = mpl.cm.ScalarMappable(cmap=vaporwave, norm=norm)
# sc.set_array([])
Zmin = None
Zmax = None
for exp_id in isotope_exps:
    data_dict = experiment_neutron_data[exp_id]
    Z = data_dict[ExperimentDataKey.PSD_HISTOGRAM]
    Zmin = Z.min() if Zmin is None or Z.min() < Zmin else Zmin
    Zmax = Z.max() if Zmax is None or Z.max() > Zmax else Zmax

norm = mpl.colors.Normalize(vmin=Zmin, vmax=Zmax)
sc = mpl.cm.ScalarMappable(cmap=vaporwave, norm=norm)
sc.set_array([])

In [ ]:
figsize = (15, 12)
fontsize = 16
histo_res = 128
contour_res = 10
angle_elev = 30
angle_rot = -60

In [ ]:
for exp_id in isotope_exps:
    data_dict = experiment_neutron_data[exp_id]
    Z = data_dict["psd_histogram_no_bg"]
    xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    isotope_name = isotope_name_lookup[exp_id]

    _x, _y = np.meshgrid(xe[:-1], ye[:-1], indexing="ij")
    xpos, ypos = _x.ravel(), _y.ravel()
    zpos = 0
    _xe, _ = np.meshgrid(xe, ye[:-1], indexing="ij")
    _, _ye = np.meshgrid(xe[:-1], ye, indexing="ij")
    _dx = np.diff(_xe, axis=0)
    _dy = np.diff(_ye, axis=1)
    dx = _dy.ravel()
    dy = _dy.ravel()
    dz = Z.ravel()
    
    # Mask bar if zero
    mask_dz = dz == 0
    xpos = xpos[~mask_dz]
    ypos = ypos[~mask_dz]
    dx = dx[~mask_dz]
    dy = dy[~mask_dz]
    dz = dz[~mask_dz]

    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection='3d')
    ax.view_init(angle_elev, angle_rot)
    # ax.contour3D(x, y, Z.T, contour_res, cmap=cmap)

    colors = vaporwave(norm(dz))

    ax.bar3d(xpos, ypos, zpos, dx, dy, dz, color=colors, shade=True)

    ax.set_title(f"{isotope_name} PSD/Energy 3D Histogram - BG removed", fontsize=fontsize+4)
    ax.set_ylabel("PSD", fontsize=fontsize)
    ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
    ax.set_zlabel("Counts", fontsize=fontsize)
    ax.set_zlim(bottom=1.1*Zmin, top=1.1*Zmax)
    ax.set_xlim(xe[0], xe[-1])

    fig.colorbar(sc, ax=ax)
    
    fig.tight_layout()
    plt.show()

In [ ]:
bg_dict = experiment_neutron_data["background"]
edges = bg_dict["edges"]
midpoints = (edges[1:] + edges[:-1]) / 2

fig, ax = plt.subplots(figsize=figsize)

for exp_id in isotope_exps:
    isotope_name = isotope_name_lookup[exp_id]
    data_dict = experiment_neutron_data[exp_id]
    Z = data_dict["histogram_no_bg"]
    ax.plot(midpoints, Z, label=isotope_name)

ax.legend()
plt.show()

In [ ]:
bg_dict = experiment_neutron_data["background"]
edges = bg_dict["edges"]
midpoints = (edges[1:] + edges[:-1]) / 2
x = np.linspace(0., 4000., len(midpoints))

def polygon_under_graph(x, y):
    """
    Construct the vertex list which defines the polygon filling the space under
    the (x, y) line graph. This assumes x is in ascending order.
    """
    return [(x[0], 0.), *zip(x, y), (x[-1], 0.)]

fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": "3d"})
all_verts = []
all_colors = []
all_indexes = []

for i, exp_id in enumerate(isotope_exps):
    isotope_name = isotope_name_lookup[exp_id]
    data_dict = experiment_neutron_data[exp_id]
    Z = data_dict["histogram_no_bg"]
    
    ys = np.full(shape=midpoints.shape, fill_value=i, dtype=int)
    exp_verts = polygon_under_graph(x, Z)
    
    all_indexes.append(i)
    all_verts.append(exp_verts)
    
    lines = ax.plot(midpoints, ys, Z, label=isotope_name)
    colors = [line.get_color() for line in lines]
    all_colors.extend(colors)

    if i == 0:
        comp_edge_x = 1000
        comp_edge_xi = np.argmax(midpoints>comp_edge_x)
        comp_edge_x_actual = midpoints[comp_edge_xi]
        comp_edge_Z = Z[comp_edge_xi]
        ax.plot([comp_edge_x_actual, comp_edge_x_actual], [0, 2], [comp_edge_Z, comp_edge_Z], "r--",linewidth=2,marker=".")

poly = mpl.collections.PolyCollection(all_verts, facecolors=all_colors, alpha=0.7)
ax.add_collection3d(poly, zs=all_indexes, zdir="y")

# ax.set_xticks(range(0,4000,500))
ax.set_yticks(all_indexes, labels=[isotope_name_lookup[id] for id in isotope_exps])
ax.zaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, _: x / 1000))
# ax.set_zticks([0, 10000, 20000, 30000, 40000, 50000])
# ax.set_zlim((0, 50000))
ax.set(zlim=(0, 50000), xlabel="ADC Channel", ylabel="Isotope", zlabel="Counts (x1000)")
ax.margins(x=0, y=0, z=0)

plt.show()

In [ ]:
bg_dict = experiment_neutron_data["background"]
# edges = bg_dict["edges"]
# midpoints = (edges[1:] + edges[:-1]) / 2
# xs = np.linspace(edges[0], edges[-1], len(midpoints)+len(edges))
midpoints = bg_dict["midpoints"]
deriv_x = bg_dict["deriv_x"]

all_calib_adcs = {
    "Na-22": [622, 2033],
    "Cs-137": [886],
    "Co-60": [1907, 2163],
    "Eu-152": [203, 383, 1055],
    "K-40": [2400]
}
all_mu_guesses = {
    "Na-22": [800, 2250],
    "Cs-137": [1050],
    "Co-60": [1950, 2250],
    "Eu-152": [400, 600, 1250, 1500, 1800, 2450],
    "K-40": [2500]
}

for exp_id in isotope_exps:
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=figsize)
    ax_top, ax_bot = axs
    ax_ul, ax_ur = ax_top
    ax_bl, ax_br = ax_bot
    
    isotope_name = isotope_name_lookup[exp_id]
    data_dict = experiment_neutron_data[exp_id]
    calib_adcs = all_calib_adcs.get(isotope_name)
    mu_guesses = all_mu_guesses.get(isotope_name)
    Z = data_dict["histogram_no_bg"]
    Z_filter = data_dict["histogram_filtered"]
    Z_interp = data_dict["histogram_interp"]
    Z_deriv = data_dict["histogram_deriv"]
    Z_deriv_filter = data_dict["histogram_deriv_filtered"]

    # ax_top.plot(xs, Z)
    # ax_bot.scatter(xs, Z_deriv, marker=".")
    ax_ul.plot(midpoints, Z, marker=".", linestyle="--")
    ax_ur.scatter(midpoints, Z_filter, marker=".")
    ax_ur.plot(xs, Z_interp(xs))
    ax_bl.plot(xs, Z_deriv)
    ax_br.plot(xs, Z_deriv_filter)

    if calib_adcs is not None:
        for adc in calib_adcs:
            ax_ur.axvline(adc, color="red", linestyle="dotted")
            ax_br.axvline(adc, color="red", linestyle="dotted")

    if mu_guesses is not None:
        for guess in mu_guesses:
            ax_ur.axvline(guess, color="blue", linestyle="dotted")
            ax_br.axvline(guess, color="blue", linestyle="dotted")

    xlim = (0, 4000)
    deriv_xlim = (750, 4000)
    deriv_ylim = (-20, 20)
    ax_ul.set(
        title="Histogram", 
        xlim=xlim, 
        xlabel="ADC Channel", 
        ylabel="Counts"
    )
    ax_ur.set(
        title="Histogram Filtered", 
        xlim=xlim, 
        xlabel="ADC Channel", 
        ylabel="Counts"
    )
    ax_bl.set(
        title="Derivative", 
        xlim=xlim, 
        # ylim=deriv_ylim, 
        xlabel="ADC Channel", 
        ylabel="Count Derivative"
    )
    ax_br.set(
        title="Derivative Filtered", 
        xlim=xlim, 
        ylim=deriv_ylim, 
        xlabel="ADC Channel", 
        ylabel="Count Derivative"
    )
    
    fig.suptitle(isotope_name)
    plt.show()

    # data_dict["histogram_filtered"] = histo_filtered
    # data_dict["histogram_interp"] = histo_interp
    # data_dict["histogram_deriv"] = histo_deriv
    # data_dict["histogram_deriv_filtered"] = histo_deriv_filtered

In [ ]:
xs = experiment_neutron_data["background"]["deriv_x"]
p0_mu_dict = {
    "Na-22": [800, 2250],
    "Cs-137": [1050],
    "Co-60": [1950, 2250],
    "Eu-152": [400, 600, 1250, 1500, 1800, 2450],
    "K-40": [2500]
}
all_mu_guesses = {
    "Na-22": [800, 2250],
    "Cs-137": [1050],
    "Co-60": [1950, 2250],
    "Eu-152": [400, 600, 1250, 1500, 1800, 2450],
    "K-40": [2500]
}
# all_calib_adcs = {
#     "Cs-137": [886],
#     "Co-60": [1907, 2163],
#     "Eu-152": [203, 383, 1055]
# }
p0_sigma_dict = {
    "Na-22": [100, 100],
    "Cs-137": [100],
    "Co-60": [50, 50],
    "Eu-152": [50, 50, 100, 50, 50, 50],
    "K-40": [100]
}
p0_A_dict = {}
for isotope_name, p0_mus in p0_mu_dict.items():
    id_matches = [k for k, v in isotope_name_lookup.items() if v == isotope_name]
    exp_id, *_ = id_matches
    data_dict = experiment_neutron_data[exp_id]
    Z_deriv_filter = data_dict["histogram_deriv_filtered"]
    p0_As = []
    for mu in p0_mus:
        mu_idx = np.argmax(xs>=mu)
        A = Z_deriv_filter[mu_idx]
        p0_As.append(A)
    p0_A_dict[isotope_name] = p0_As

In [ ]:
xs = experiment_neutron_data["background"]["deriv_x"]
midpoints = experiment_neutron_data["background"]["midpoints"]

for exp_id in isotope_exps:
    data_dict = experiment_neutron_data[exp_id]
    isotope_name = isotope_name_lookup[exp_id]

    Z_filter = data_dict["histogram_filtered"]
    Z_deriv_filter = data_dict["histogram_deriv_filtered"]
    
    p0_mus = p0_mu_dict[isotope_name]
    p0_sigmas = p0_sigma_dict[isotope_name]
    p0_As = p0_A_dict[isotope_name]
    p0s = zip(p0_mus, p0_sigmas, p0_As)
    # calib_adcs = all_calib_adcs[isotope_name]

    all_params = []
    all_fit_xs = []
    for p0 in p0s:
        p0_mu, p0_sigma, *_ = p0
        
        lo = p0_mu - 3 * p0_sigma
        hi = p0_mu + 3 * p0_sigma
        lo_idx = np.argmax(xs>=lo)
        hi_idx = np.argmax(xs>hi)
        
        xs_fit = xs[lo_idx:hi_idx]
        Z_fit = Z_deriv_filter[lo_idx:hi_idx]
        
        params, _ = curve_fit(gaussian, xs_fit, Z_fit, p0=p0)
        
        all_fit_xs.append(xs_fit)
        all_params.append(params)
    
    data_dict["compton_fits"] = all_params
    data_dict["compton_xs"] = all_fit_xs
    
    # plot deriv with gaussians
    fig, axs = plt.subplots(nrows=2, figsize=figsize)
    ax_top, ax_bot = axs

    # ylim = None
    # ylim = (-50, 50)
    # ylim = (-20, 20)
    ylim = (-5, 5)
    ax_top.plot(midpoints, Z_filter, label="Filtered Histogram")
    ax_bot.plot(xs, Z_deriv_filter, label="Filtered Derivative")
    for i, (fit_xs, fit_params) in enumerate(zip(all_fit_xs, all_params)):
        fit_mu, fit_sigma, fit_A = fit_params
        ax_bot.plot(fit_xs, gaussian(fit_xs, *fit_params), label=f"Compton edge {i+1}")
        if ylim is None or ylim[0] < fit_A < ylim[1]:
            ax_bot.text(
                fit_mu, 
                fit_A, 
                f"mu={fit_mu:.2f}\nsigma={fit_sigma:.2f}", 
                horizontalalignment="center", 
                verticalalignment="top", 
                bbox={"boxstyle": "round", "color": "white", "alpha": 0.7}
            )
        ax_top.axvline(fit_mu, linestyle="--", color="black")
    # for calib_channel in calib_adcs:
    #     ax_top.axvline(calib_channel, linestyle="dotted", color="red")
    #     ax_bot.axvline(calib_channel, linestyle="dotted", color="red")
    ax_top.set(title="Histogram", xlabel="ADC Channel", ylabel="Counts")
    ax_bot.set(title="Derivative", xlabel="ADC Channel", ylabel="Derivative Counts", ylim=ylim)
    fig.suptitle(isotope_name)
    plt.show()

In [ ]:
def light_energy(E_gamma):
    rest_mass = 0.510998950692
    return (2*E_gamma*E_gamma)/(rest_mass+2*E_gamma)

all_comp_edge_energies = {
    "Na-22": [0.55, 1.28],
    "Cs-137": [0.66],
    "Eu-152": [0.25, 0.34, 0.78, 0.96, 1.11, 1.41],
    "Co-60": [1.17, 1.33],
    "K-40": [1.46],
    "AmBe": [4.44],
}

calib_data: tuple[str, float, float] = []

for exp_id in isotope_exps:
    data_dict = experiment_neutron_data[exp_id]
    isotope_name = isotope_name_lookup[exp_id]
    energies = all_comp_edge_energies[isotope_name]
    Ls = [light_energy(energy) for energy in energies]
    fit_params = data_dict["compton_fits"]
    fit_mus = [fit_mu for fit_mu, *_ in fit_params]
    isotope_calib_data = list(zip(repeat(isotope_name), fit_mus, Ls))
    calib_data.extend(isotope_calib_data)
calib_data = sorted(calib_data, key=lambda x: x[2])
calib_data

In [ ]:
_, fit_mus, Ls = zip(*calib_data)
calib_fit_params, calib_fit_cov = np.polyfit(Ls, fit_mus, 1, cov=True)
calib_fit_error = np.sqrt(np.diag(calib_fit_cov))
calib_fit_params

In [ ]:
old_calib_data = [
    ("Cs-137", 0.6650, 886),
    ("Co-60", 1.1700, 1907),
    ("Co-60", 1.3300, 2163),
    ("Eu-152", 0.1220, 203),
    ("Eu-152", 0.3440, 383),
    ("Eu-152", 0.7790, 1055),
    ("Na-22", 0.5110, 622),
    ("Na-22", 1.2750, 2033)
]
old_calib_data = [(name, light_energy(energy), adc) for name, energy, adc in old_calib_data]
old_calib_data = sorted(old_calib_data, key=lambda x: x[2])
old_calib_data

In [ ]:
_, old_Ls, old_adcs = zip(*old_calib_data)
old_calib_fit_params, old_calib_fit_cov = np.polyfit(old_Ls, old_adcs, 1, cov=True)
old_calib_fit_error = np.sqrt(np.diag(old_calib_fit_cov))
old_calib_fit_params

In [ ]:
# TODO plot calibration curve
minL = None
maxL = None
fig, ax = plt.subplots(figsize=figsize)
for isotope_name in isotope_name_lookup.values():
    isotope_calib_data = [x for x in calib_data if x[0] == isotope_name]
    _, fit_mus, Ls = zip(*isotope_calib_data)
    minL = min(Ls) if minL is None or min(Ls) < minL else minL
    maxL = max(Ls) if maxL is None or max(Ls) > maxL else maxL
    ax.scatter(Ls, fit_mus, label=isotope_name)

    old_isotope_calib_data = [x for x in old_calib_data if x[0] == isotope_name]
    if len(old_isotope_calib_data) <= 0:
        continue
    _, old_Ls, old_adcs = zip(*old_isotope_calib_data)
    minL = min(old_Ls) if minL is None or min(old_Ls) < minL else minL
    maxL = max(old_Ls) if maxL is None or max(old_Ls) > maxL else maxL
    ax.scatter(old_Ls, old_adcs, label=f"{isotope_name} (Old Calib.)", marker="x")

x = np.linspace(minL, maxL, num=100)
m, b = calib_fit_params
m_var, b_var = calib_fit_error
y = m * x + b
ax.plot(x, y, label="Best fit")

m_old, b_old = old_calib_fit_params
m_var_old, b_var_old = old_calib_fit_error
y_old = m_old * x + b_old
ax.plot(x, y_old, label="Best fit (Old Calib.)")

ax.text(
    0.995,
    0.01,
    f"New Calibration\nm={m:.2f}+/-{m_var:.2f}\nb={b:.2f}+/-{b_var:.2f}",
    transform=ax.transAxes,
    horizontalalignment="right"
)
ax.text(
    0.995,
    0.07,
    f"Old Calibration\nm={m_old:.2f}+/-{m_var_old:.2f}\nb={b_old:.2f}+/-{b_var_old:.2f}",
    transform=ax.transAxes,
    horizontalalignment="right"
)
ax.set(title="Calibration Curve", xlabel="L (MeVee)", ylabel="Amplitude (V)")
ax.legend()
plt.show()

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()

In [ ]:
bg_dict = experiment_neutron_data["background"]
xs = bg_dict["midpoints"]
order = 6
period = xs[1] - xs[0]
fs = 1/period
cutoff = 0.02

for exp_id in isotope_exps:
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=figsize)
    axs_top, axs_bot = axs
    ax_ul, ax_ur = axs_top
    ax_bl, ax_br = axs_bot
    
    isotope_name = isotope_name_lookup[exp_id]
    data_dict = experiment_neutron_data[exp_id]
    # Z_interp = data_dict["histogram_interp"]
    # Z = Z_interp(xs)
    Z = data_dict["histogram_no_bg"]
    Z_fft = rfft(Z)
    freqs = rfftfreq(Z.shape[-1], d=period)
    ax_ul.plot(xs, Z, label="Original")
    ax_ul.plot(xs, butter_lowpass_filter(Z, cutoff, fs, order), label="Filtered")
    ax_bl.plot(freqs, np.abs(Z_fft))
    for cutoff in np.linspace(0.01, 0.04, num=4):
        Z_filter = butter_lowpass_filter(Z, cutoff, fs, order)
        Z_fft_filter = rfft(Z_filter)
        ax_ur.plot(xs, Z_filter, label=cutoff)
        ax_br.plot(freqs, np.abs(Z_fft_filter), label=cutoff)
    # Z_deriv = Z_interp.derivative()(xs)
    # Z_lowpass = butter_lowpass_filter(Z_deriv, cutoff, fs, order)
    # Z_deriv = data_dict["histogram_deriv"]
    # Z_lowpass = data_dict["histogram_deriv_lowpass"]
    # ax_top.plot(xs, Z)
    ax_ul.set_title("Histogram")
    ax_bl.set_title("FFT")
    ax_ur.set_title("Filtered")
    ax_br.set_title("FFT")
    ax_ul.legend()
    ax_ur.legend()
    ax_br.legend()

    # fft_limits = (0, 0.1)
    # ax_bl.set(xlim=fft_limits)
    # ax_br.set(xlim=fft_limits)
    # ax_ul.set(xlim=(0, 1000), ylim=(15000, 20000))
    ylimits = (0, 100000)
    ax_bl.set_ylim(ylimits)
    ax_br.set_ylim(ylimits)
    # ax.legend()
    fig.suptitle(isotope_name)
    plt.show()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# Separate neutron and gamma events
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]

    n_classify_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    neutrons_only = psd_report.query(n_classify_col_name).copy()
    gamma_only = psd_report.query(f"~{n_classify_col_name}").copy()
    data_dict[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    data_dict[ExperimentDataKey.GAMMA_ONLY] = gamma_only

### Non-Neutron Data Processing

In [ ]:
# process reactor data files
# stored in reactor_data
# File name format: Device Param 00x
# If same device/param, but different numbers, should be merged

data_file_pattern = re.compile(r"([a-zA-Z ]+) (\d+)")
for exp_name, data_dict in experiment_neutron_data.items():
    reactor_data_folder = get_reactor_data_root(exp_name)
    time_col_name = NonReactorDataframeColumn.TIME.value
    data_col_name = NonReactorDataframeColumn.DATA.value
    units_col_name = NonReactorDataframeColumn.UNITS.value
    
    if not reactor_data_folder.is_dir():
        continue

    reactor_data_files = defaultdict(list)
    for file in reactor_data_folder.iterdir():
        match = data_file_pattern.match(file.name)
        if match and len(match.groups()) > 0:
            data_source = match.group(1)
            if isinstance(data_source, str):
                reactor_data_files[data_source].append(file)

    reactor_data = {}
    for data_source, files in reactor_data_files.items():
        file_dfs = [
            pd.read_csv(
                file,
                header=0,
                dtype=str,
                encoding='cp1252',
                names=[time_col_name, data_col_name, units_col_name]
            ) for file in sorted(files)]
        for df in file_dfs:
            df[time_col_name] = pd.to_datetime(
                df[time_col_name], utc=True)
        df = pd.concat(file_dfs, ignore_index=True)
        reactor_data[data_source] = df

    data_dict[ExperimentDataKey.REACTOR_DATA] = reactor_data

## Data Binning

In [ ]:
# Create bins
for exp_name, data_dict in experiment_neutron_data.items():
    neutron_report = data_dict[ExperimentDataKey.NEUTRONS_ONLY]

    event_time_col = DetectorDataframeColumn.EVENT_TIME.value
    start_time = neutron_report[event_time_col].min()
    end_time = neutron_report[event_time_col].max()
    timetag_clock_bins = pd.date_range(
        start=start_time, end=end_time, freq=bin_string)
    data_dict[ExperimentDataKey.TIME_BIN_EDGES] = timetag_clock_bins

In [ ]:
# Bin neutron data
for exp_name, data_dict in experiment_neutron_data.items():
    neutron_report = data_dict[ExperimentDataKey.NEUTRONS_ONLY]
    time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]

    time_col_name = DetectorDataframeColumn.EVENT_TIME.value
    time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
    count_col_name = BinningDataframeColumn.COUNT.value
    count_error_col_name = BinningDataframeColumn.COUNT_ERROR.value
    bin_mid_col_name = BinningDataframeColumn.BIN_MIDPOINT.value
    n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
    n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value

    start_time = time_bins[0]

    binned_neutrons = get_time_cut(
        neutron_report, time_col_name, time_bins)
    binned_neutrons = neutron_report.groupby(
        time_bin_col_name, as_index=True) \
        .size() \
        .to_frame() \
        .copy()
    binned_neutrons.columns = [count_col_name]
    binned_neutrons[count_error_col_name] = np.sqrt(
        binned_neutrons[count_col_name]
    )

    binned_neutron_time_bins = binned_neutrons.index.to_series()
    midpoints = binned_neutron_time_bins.apply(lambda x: x.mid)
    durations = binned_neutron_time_bins.apply(
        lambda x: x.length.total_seconds()
    )

    binned_neutrons[bin_mid_col_name] = midpoints
    binned_neutrons = bin_midpoint_time_to_seconds(binned_neutrons, start_time)

    binned_neutrons[n_rate_col_name] = (
        binned_neutrons[count_col_name] / durations)
    binned_neutrons[n_error_col_name] = (
        binned_neutrons[count_error_col_name] / durations)
    binned_neutrons = binned_neutrons.drop(
        [count_col_name, count_error_col_name],
        axis=1
    ) \
        .copy()
    data_dict[ExperimentDataKey.BINNED_NEUTRONS] = binned_neutrons

In [ ]:
# Bin gamma data
for exp_name, data_dict in experiment_neutron_data.items():
    gamma_report = data_dict[ExperimentDataKey.GAMMA_ONLY]
    time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]

    time_col_name = DetectorDataframeColumn.EVENT_TIME.value
    time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
    count_col_name = BinningDataframeColumn.COUNT.value
    count_error_col_name = BinningDataframeColumn.COUNT_ERROR.value
    bin_mid_col_name = BinningDataframeColumn.BIN_MIDPOINT.value
    g_rate_col_name = BinningDataframeColumn.GAMMA_RATE.value
    g_error_col_name = BinningDataframeColumn.GAMMA_RATE_ERROR.value

    start_time = time_bins[0]

    binned_gamma = get_time_cut(
        gamma_report, time_col_name, time_bins)
    binned_gamma = gamma_report.groupby(
        time_bin_col_name, as_index=True) \
        .size() \
        .to_frame() \
        .copy()
    binned_gamma.columns = [count_col_name]
    binned_gamma[count_error_col_name] = np.sqrt(
        binned_gamma[count_col_name])

    binned_gamma_time_bins = binned_gamma.index.to_series()
    midpoints = binned_gamma_time_bins.apply(lambda x: x.mid)
    durations = binned_gamma_time_bins.apply(
        lambda x: x.length.total_seconds())

    binned_gamma[bin_mid_col_name] = midpoints
    binned_gamma = bin_midpoint_time_to_seconds(binned_gamma, start_time)

    binned_gamma[g_rate_col_name] = (
        binned_gamma[count_col_name] / durations)
    binned_gamma[g_error_col_name] = (
        binned_gamma[count_error_col_name] / durations)
    binned_gamma = binned_gamma.drop(
        [count_col_name, count_error_col_name],
        axis=1
    ).copy()
    data_dict[ExperimentDataKey.BINNED_GAMMA] = binned_gamma

In [ ]:
# bin gamma energy spectrum

def make_index_converter(
    start_time: pd.Timestamp
) -> Callable:
    def index_converter(
        category: pd.Interval
    ) -> pd.Interval:
        cat_start = category.left
        cat_end = category.right
        start_seconds = (cat_start - start_time).total_seconds()
        end_seconds = (cat_end - start_time).total_seconds()
        return pd.Interval(start_seconds, end_seconds, closed=category.closed)

    return index_converter


for exp_name, data_dict in experiment_neutron_data.items():
    gamma_report = data_dict[ExperimentDataKey.GAMMA_ONLY]
    time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]
    energy_bins = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]

    time_col_name = DetectorDataframeColumn.EVENT_TIME.value
    time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
    eng_bin_col_name = BinningDataframeColumn.ENERGY_BIN.value
    count_col_name = BinningDataframeColumn.COUNT.value

    start_time = time_bins[0]

    binned_gamma_spectrum = get_time_cut(
        gamma_report, time_col_name, time_bins)
    energy_cut, energy_bins = pd.cut(
        binned_gamma_spectrum[calibrated_energy_column.value],
        bins=energy_bins,
        retbins=True
    )
    binned_gamma_spectrum[eng_bin_col_name] = energy_cut
    binned_gamma_spectrum = binned_gamma_spectrum.groupby(
        [time_bin_col_name, eng_bin_col_name],
        as_index=True
    ) \
        .size() \
        .to_frame() \
        .copy()
    binned_gamma_spectrum.columns = [count_col_name]
    binned_gamma_spectrum = binned_gamma_spectrum.reset_index(level=1)
    binned_gamma_spectrum = binned_gamma_spectrum.pivot_table(
        values=count_col_name,
        index=binned_gamma_spectrum.index,
        columns=eng_bin_col_name
    )
    
    time_index = binned_gamma_spectrum.index
    start_time = time_index[0].left
    convert_categories = make_index_converter(start_time)
    time_index = time_index.map(convert_categories)
    binned_gamma_spectrum.index = time_index

    data_dict[ExperimentDataKey.GAMMA_ENERGY_SPECTRUM] = binned_gamma_spectrum
    data_dict[ExperimentDataKey.GAMMA_ENERGY_BIN_EDGES] = energy_bins

In [ ]:
# process and bin reactor data

def normalize_units(value, unit, to_unit):
    '''
    Converts Series of values to desired units

    value: measured value
    units: units of measured value
    to_unit: unit to convert to

    returns Series of values converted to desired unit
    '''
    try:
        return Quantity(value, unit).ito(to_unit).magnitude
    except AttributeError:
        return value


for exp_name, data_dict in experiment_neutron_data.items():
    time_col_name = NonReactorDataframeColumn.TIME.value
    data_col_name = NonReactorDataframeColumn.DATA.value
    units_col_name = NonReactorDataframeColumn.UNITS.value
    norm_data_col_name = NonReactorDataframeColumn.NORMALIZED_DATA.value
    norm_units_col_name = NonReactorDataframeColumn.NORMALIZED_UNITS.value

    if (
        reactor_data := data_dict.get(ExperimentDataKey.REACTOR_DATA)
    ) is not None:
        binned_reactor_data = {}
        time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]

        for data_source, reactor_param_df in reactor_data.items():
            try:
                reactor_param_df[data_col_name] = reactor_param_df[
                    data_col_name
                ].astype(float)
            except ValueError:
                continue  # skip if not numeric

            units_counts = reactor_param_df[units_col_name].value_counts()
            main_unit = units_counts.idxmax()
            if units_counts.size > 1:
                # determine most frequent
                reactor_param_df[norm_data_col_name] = reactor_param_df.apply(
                    lambda row: normalize_units(
                        row[data_col_name],
                        row[units_col_name],
                        main_unit
                    ),
                    axis=1
                )
                reactor_param_df = reactor_param_df.assign(
                    **{norm_units_col_name: lambda _: main_unit}
                )
            else:
                reactor_param_df[norm_data_col_name] = reactor_param_df[
                    data_col_name]
                reactor_param_df[norm_units_col_name] = reactor_param_df[
                    units_col_name]

            binned_reactor_param_df = bin_non_neutron_data(
                reactor_param_df,
                time_bins,
                norm_data_col_name,
                [f"Average {data_source} ({main_unit})",
                 f"{data_source} error ({main_unit})"]
            )
            binned_reactor_data[data_source] = binned_reactor_param_df
        data_dict[ExperimentDataKey.BINNED_REACTOR_DATA] = binned_reactor_data

In [ ]:
# Merge binned data
for exp_name, data_dict in experiment_neutron_data.items():
    time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
    bin_mid_col_name = BinningDataframeColumn.BIN_MIDPOINT.value
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value

    binned_dfs = []
    binned_dfs.append(data_dict[ExperimentDataKey.BINNED_NEUTRONS])
    binned_dfs.append(data_dict[ExperimentDataKey.BINNED_GAMMA])
    binned_reactor_data = data_dict.get(ExperimentDataKey.BINNED_REACTOR_DATA)
    if binned_reactor_data is not None:
        binned_reactor_param_dfs = binned_reactor_data.values()
        for binned_reactor_param_df in binned_reactor_param_dfs:
            binned_dfs.append(binned_reactor_param_df)
    merged_df = reduce(
        lambda df1, df2: pd.merge(
            df1, df2, how='left', on=[
                time_bin_col_name, bin_mid_col_name, bin_time_col_name
            ]
        ),
        binned_dfs
    )
    data_dict[ExperimentDataKey.ALL_BINNED_DATA] = merged_df

## Export and Display

In [ ]:
# clear old data from root, and make new analysis folder
for exp_name in experiment_neutron_data.keys():
    root = get_report_root(exp_name)
    analysis_root = root / analysis_timestamp
    for file in root.iterdir():
        if file.is_file() and not file.is_dir():
            file.unlink()
    analysis_root.mkdir(parents=True, exist_ok=True)

In [ ]:
# add analysis setting file to experiment root and analysis folder
for exp_name in experiment_neutron_data.keys():
    root = get_report_root(exp_name)
    analysis_root = root / analysis_timestamp
    file_name = f"{exp_name}_analysis_settings_{analysis_timestamp}.json"
    with open(root / file_name, 'w') as root_settings_file:
        json.dump(overall_settings, root_settings_file)
    # with open(analysis_root / file_name, 'w') as analysis_settings_file:
    #     json.dump(overall_settings, analysis_settings_file)
    try:
        shutil.copy(root / file_name, analysis_root / file_name)
    except shutil.SameFileError:
        pass

In [ ]:
# Export as CSV
for exp_name, data_dict in experiment_neutron_data.items():
    all_binned_data = data_dict[ExperimentDataKey.ALL_BINNED_DATA]
    root = get_report_root(exp_name)
    analysis_root = root / analysis_timestamp
    file_name = f"{exp_name}_data_{bin_length}s_bin.csv"
    root_path = root / file_name
    analysis_path = analysis_root / file_name
    all_binned_data.to_csv(root_path, index=False)
    try:
        shutil.copy(root_path, analysis_path)
    except shutil.SameFileError:
        pass
    print(f"Experiment {exp_name} saved to:")
    print(f"    - {root_path}")
    print(f"    - {analysis_path}")

In [ ]:
for exp_name, data_dict in experiment_neutron_data.items():
    binned_gamma_spectrum = data_dict[ExperimentDataKey.GAMMA_ENERGY_SPECTRUM]
    root = get_report_root(exp_name)
    analysis_root = root / analysis_timestamp
    file_name = (f"{exp_name}_gamma_spectrum_{bin_length}s_time_bin.csv")
    root_path = root / file_name
    analysis_path = analysis_root / file_name
    binned_gamma_spectrum.to_csv(root_path)
    try:
        shutil.copy(root_path, analysis_path)
    except shutil.SameFileError:
        pass
    print(f"Gamma spectrum for {exp_name} saved to:")
    print(f"    - {root_path}")
    print(f"    - {analysis_path}")

In [ ]:
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
    
    root = get_report_root(exp_name)
    analysis_root = root / analysis_timestamp
    file_name = f"{exp_name}_event_psd_energy.csv"
    root_path = root / file_name
    analysis_path = analysis_root / file_name
    
    columns = [
        calibrated_energy_column.value,
        DetectorDataframeColumn.PSD.value,
        DetectorDataframeColumn.NEW_N_CLASS.value
    ]
    headers = ['Energy (MeVee)', 'PSD', 'Is Neutron?']
    psd_report.to_csv(
        root_path,
        index=False,
        columns=columns,
        header=headers
    )
    try:
        shutil.copy(root_path, analysis_path)
    except shutil.SameFileError:
        pass
    print(f"Event energy/PSD data for {exp_name} saved to:")
    print(f"    - {root_path}")
    print(f"    - {analysis_path}")

## Diagnostics Display

In [ ]:
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
    fig, ax = plot_scatter(
        psd_report[calibrated_energy_column.value],
        psd_report[DetectorDataframeColumn.PSD.value]
    )
    ax.set_xlabel("Energy [MeVee]", fontsize=14)  # Update x-axis label
    ax.set_ylabel("PSD", fontsize=14)
    
    ax.set_title(
        f"{exp_name} PSD/Energy Graph",
        ha='center',
        fontsize=20
    )
    output_path = get_report_root(exp_name) / f"{exp_name} PSD graph.png"
    fig.savefig(output_path)
    plt.show()

In [ ]:
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
    fom_results = data_dict[ExperimentDataKey.FOM_RESULTS]
    x_bin_edges = helpers.get_midpoints_from_min_max_series(
        fom_results[SliceFitDataframeColumn.SLICE_ENERGY_MINIMUM.value],
        fom_results[SliceFitDataframeColumn.SLICE_ENERGY_MAXIMUM.value],
        fom_results.index
    )
    gamma_mu = fom_results.mu1
    gamma_sigma = fom_results.sigma1
    neutron_sigma = fom_results.sigma2
    
    window_sigma = 5*gamma_sigma
    fom_sigma = 3*(gamma_sigma+neutron_sigma)
    
    fig, ax = plot_scatter(
        psd_report[calibrated_energy_column.value],
        psd_report[DetectorDataframeColumn.PSD.value]
    )
    dot_size = 8
    ax.scatter(
        x_bin_edges,
        window_sigma,
        marker=".",
        linewidths=0,
        s=dot_size,
        label="5 x gamma sigma"
    )
    ax.scatter(
        x_bin_edges,
        fom_sigma,
        marker="o",
        linewidths=0,
        s=dot_size,
        label="3 x sigma sum"
    )
    ax.set_xlabel("Energy [MeVee]", fontsize=14)  # Update x-axis label
    ax.set_ylabel("PSD", fontsize=14)
    # Add a title
    ax.set_title(
        f"{exp_name} PSD/Energy Graph",
        ha='center',
        fontsize=20
    )
    # output_path = get_report_root(exp_name) / f"{exp_name} PSD graph.png"
    # fig.savefig(output_path)
    plt.legend()
    plt.show()

In [ ]:
### For named color options (and examples), check https://matplotlib.org/stable/gallery/color/named_colors.html#css-colors
# (For more options, check https://matplotlib.org/stable/gallery/color/named_colors.html#css-colors)
# (Any text option (in quotes) can be used)
for exp_name, data_dict in experiment_neutron_data.items():
    binned_neutrons = data_dict[ExperimentDataKey.ALL_BINNED_DATA]

    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
    n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value
    
    zeroed_bins = binned_neutrons[bin_time_col_name] / 60
    rates = binned_neutrons[n_rate_col_name]
    rate_errors = binned_neutrons[n_error_col_name]

    color_block_settings = []
    add_color_blocks = helpers.get_input_with_default("Do you want to add color blocks? [y/N] >", "n", str)
    if add_color_blocks.lower() == "y":
        color_blocks_count = helpers.get_input_required("How many color blocks do you want to add? (minimum 1) >", (1, None), int)
        for i in range(color_blocks_count):
            print(f"For block {i+1}:")
            block_start = helpers.get_input_required("Where should the block start (in minutes elapsed)? >", (0, None), int)
            block_end = helpers.get_input_required("Where should the block end (in minutes elapsed)? >", (block_start, None), int)
            block_color = input("What color should the block be? >")
            block_settings = {"start": block_start, "end": block_end, "color": block_color}
            color_block_settings.append(block_settings)
    
    fig, ax = plt.subplots(figsize=(12, 8), dpi=300)
    dot_size = 8
    
    ax.errorbar(
        zeroed_bins,
        rates,
        yerr=rate_errors,
        fmt=".",
        linestyle='',
        markersize=dot_size,
        capsize=dot_size
    )
    ax.set_xlabel("Time [minutes]", fontsize=14)  # Update x-axis label
    ax.set_ylabel("Neutron count rate [1/s]", fontsize=14)
    ax.tick_params(labelsize=12)
    ax.set_ylim(-5, 205)
    ax.set_xlim(0, 145)
    
    for block_settings in color_block_settings:
        ax.axvspan(block_settings["start"], block_settings["end"], color=block_settings["color"], alpha=0.2)

    # Add a title above the plot
    fig.text(
        0.5,
        0.90,
        f"{exp_name} neutron count rate over time (Dwell time {bin_length}s)",
        ha='center',
        fontsize=20
    )

    # Save the plot
    output_path = (
        get_report_root(exp_name)
        / f"{exp_name} Count rates {bin_length}s dwell.png"
    )
    fig.savefig(output_path)

    # Show the plot (optional)
    plt.show()

In [ ]:
# import matplotlib as mpl
HISTOGRAM_RES = 1024
COUNT_LIMIT = 20

for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
    borders = data_dict[ExperimentDataKey.BORDERS]
    
    fig, ax = plot_classification(
        psd_report,
        borders,
        exp_name,
        DetectorDataframeColumn.NEW_N_CLASS,
        calibrated_energy_column,
        count_limit=COUNT_LIMIT,
        colormap_name="seismic"
    )

    output_path = (
        get_report_root(exp_name)
        / f"{exp_name} Neutron Classification.png"
    )
    fig.savefig(output_path)

    plt.show()

In [ ]:
fom_to_sigma = 2 * sqrt(2 * log(2))

for exp_name, data_dict in experiment_neutron_data.items():
    fom_results = data_dict[ExperimentDataKey.FOM_RESULTS]
    slice_mid_energy = helpers.get_midpoints_from_min_max_series(
        fom_results[SliceFitDataframeColumn.SLICE_ENERGY_MINIMUM.value],
        fom_results[SliceFitDataframeColumn.SLICE_ENERGY_MAXIMUM.value],
        fom_results.index
    ).to_numpy(copy=True)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(
        slice_mid_energy,
        fom_results[SliceFitDataframeColumn.FOM.value]
    )
    ax.set_xlabel("Energy [MeVee]", fontsize=14)  # Update x-axis label
    ax.set_ylabel("FOM", fontsize=14)
    # Add a title
    ax.set_title(
        f"{exp_name} FOM Graph",
        ha='center',
        fontsize=20
    )
    ax.hlines(1.27, slice_mid_energy[0], slice_mid_energy[-1], "r", ls="--")

    output_path = (
        get_report_root(exp_name)
        / f"{exp_name} Energy vs FOM.png"
    )
    fig.savefig(output_path)
    plt.show()

In [ ]:
# histogram contour plot (vaporwave island)
cmap = plt.colormaps["nipy_spectral"]
figsize = (12, 12)
fontsize = 16
histo_res = 128
contour_res = 100
angle_elev = 30
angle_rot = -60

for exp_name, data_dict in experiment_neutron_data.items():
    Z = data_dict[ExperimentDataKey.PSD_HISTOGRAM]
    xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection='3d')
    x, y = np.meshgrid(xe[:-1], ye[:-1])

    ax.view_init(angle_elev, angle_rot)
    ax.contour3D(x, y, Z.T, contour_res, cmap=cmap)
    ax.set_title(f"{exp_name} PSD/Energy 3D Histogram", fontsize=fontsize+4)
    ax.set_ylabel("PSD", fontsize=fontsize)
    ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
    ax.set_zlabel("Counts", fontsize=fontsize)

    output_path = (
        get_report_root(exp_name)
        / f"{exp_name} PSD 3D Histogram.png"
    )
    fig.savefig(output_path)

    plt.show()

In [ ]:
cmap = plt.colormaps["nipy_spectral"]
figsize = (12, 12)
fontsize = 16
contour_res = 50
angle_elev = 30
angle_rot = -60
ls = LightSource(270, 45)

for exp_name, data_dict in experiment_neutron_data.items():
    Z = data_dict[ExperimentDataKey.GAMMA_ENERGY_SPECTRUM].to_numpy()
    xe = data_dict[ExperimentDataKey.GAMMA_ENERGY_BIN_EDGES]
    # TODO find x edge closest to high cutoff (0.5)
    time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]
    start_time = time_bins[0]
    ye = (time_bins - start_time).total_seconds() / 3600
    xmid = (xe[1:] + xe[:-1]) / 2
    ymid = (ye[1:] + ye[:-1]) / 2
    # xmid = xe[:-1]
    # ymid = ye[:-1]

    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection='3d')
    x, y = np.meshgrid(xmid, ymid)

    ax.view_init(angle_elev, angle_rot)
    rgb = ls.shade(Z, cmap=cmap, blend_mode='soft')
    # ax.contour3D(x, y, Z, contour_res, cmap=cmap)
    ax.plot_surface(
        x, y, Z,
        rstride=1, cstride=1,
        facecolors=rgb, linewidth=0,
        antialiased=False, shade=False)
    ax.set_title(f"{exp_name} PSD/Energy 3D Histogram", fontsize=fontsize+4)
    ax.set_ylabel("Time (hours)", fontsize=fontsize)
    ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
    ax.set_zlabel("Counts", fontsize=fontsize)

    output_path = (
        get_report_root(exp_name)
        / f"{exp_name} Gamma Spectrum 3D Histogram.png"
    )
    fig.savefig(output_path)

    plt.show()

In [ ]:
for exp_name, data_dict in experiment_neutron_data.items():
    gamma_energy_spectrum_df = data_dict[ExperimentDataKey.GAMMA_ENERGY_SPECTRUM]
    do_spectrum = input(f"Do you want a gamma spectrum for {exp_name}? [y/n] ")
    if do_spectrum.lower() not in ["y", "yes"]:
        continue
    elapsed_str = input(
        "At what time (elapsed hours) do you want to get the spectrum? ")
    try:
        spectrum_time_elapsed = float(elapsed_str)
    except ValueError:
        print(f"The value {elapsed_str} was not a valid decimal number")
        continue
    time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]
    start_time = time_bins[0]
    spectrum_timestamp = start_time + timedelta(hours=spectrum_time_elapsed)

    matching_intervals = [
        interval for interval
        in gamma_energy_spectrum_df.index.categories
        if spectrum_timestamp in interval
    ]
    if len(matching_intervals) == 0:
        print((
            f"The given time ({spectrum_time_elapsed} hrs) " +
            "has no match in this experiment"
        ))
        continue
    matching_interval = matching_intervals[0]
    gamma_test_spectrum = gamma_energy_spectrum_df[
        gamma_energy_spectrum_df.index == matching_interval]

    gamma_energy_bins = gamma_test_spectrum.T.index.to_series()
    midpoints = gamma_energy_bins.apply(lambda x: x.mid)
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(midpoints, gamma_test_spectrum.T, s=1)
    ax.set_title(
        f"Gamma Energy Spectrum - {exp_name} @ {spectrum_time_elapsed} hrs")
    ax.set_xlabel("Energy (MeVee)")
    ax.set_ylabel("Count")

    plt.show()

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()